In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


## LLMAG Master Pipeline: Agent-Based Automatic Geological Generalization

This notebook implements the LLMAG master pipeline for automatic geological generalization of Pakistan copper-related geological data. The workflow combines an optional LLM-based semantic normalization step with deterministic GIS processing to produce reproducible generalized geology layers, copper-cluster outputs, and summary tables.

### How the workflow operates

The LLMAG controller acts as a workflow orchestrator. It does not replace GIS analysis. Instead, it divides the overall task into ordered steps and sends each step to the appropriate module.

1. **Read structured inputs**
   The pipeline loads the geology shapefile, tectonic boundaries, faults, control table, and copper occurrence data.

2. **Optional semantic normalization**
   If an API key is available, DeepSeek is used only to normalize auxiliary geological unit information and support unit-label standardization.

3. **Merge semantic support with the control table**
   The semantic output is merged with the existing CSV mapping to build a final standardized unit-to-category table. If the LLM step is unavailable, the workflow automatically falls back to the control CSV.

4. **Deterministic tectonic-guided geological generalization**
   Using GeoPandas operations, geological polygons are intersected with tectonic units and dissolved by tectonic unit and geological category to produce the generalized geology layer.

5. **Copper clustering and spatial preparation**
   Copper points are cleaned, filtered, and grouped using Gaussian Mixture Modelling (GMM), and the main spatial objects used by the notebook are prepared.

6. **Output generation**
   The prepared objects are then used by the downstream map and table blocks.

### Core objects prepared by the pipeline

After execution, the notebook prepares the main objects used in the later result sections:

* `gdf_geology_merged`
* `gdf_tectonic_pk`
* `gdf_faults_pk`
* `copper_gdf`
* `mapping_df_final`

### What “agent” means in this notebook

In LLMAG, the agent is a controlled orchestration layer that:

* breaks the workflow into ordered tasks,
* assigns semantic normalization to the LLM module,
* assigns spatial operations to deterministic GIS modules,
* preserves traceability and reproducibility.

The LLM is **not** used for spatial analysis or direct map generation. Its role is limited to semantic normalization, while all geological generalization, overlay, dissolve, clipping, clustering, and export operations remain deterministic.

### Notes

* Optional semantic support can be enabled when available; otherwise the workflow runs fully with the control CSV and deterministic GIS processing.
* If the API key is unavailable, the workflow falls back to the CSV-based mapping.
* All spatial processing and output generation remain deterministic.
* The notebook preserves the downstream plotting and result workflow.




In [7]:
# -*- coding: utf-8 -*-
"""
LLMAG Agent-Based Master Pipeline for Automatic Geological Generalization

Main idea:
- structured geological inputs are read first
- optional semantic text can be normalized by DeepSeek
- semantic support is merged with the control table
- deterministic GIS overlay/dissolve performs the real generalization
- copper points are clustered and core outputs are exported
"""

import os
import re
import json
import warnings
from time import time

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely.geometry import box
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

try:
    import requests
except Exception:
    requests = None

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ============================================================
# INPUT PATHS
# ============================================================
GEOLOGY_SHP  = "/kaggle/input/datasets/atifbilal/pakistan-lithology-mergedpakistan-lithology-merged/Export_Output_3.shp"
CONTROL_CSV  = "/kaggle/input/datasets/atifbilal/lithology-finalized-with-age-csv/Lithology_finalized_with_age_category_midage_categorymid.csv"
COPPER_XLSX  = "/kaggle/input/datasets/atifbilal/merged-shp-copepr-data-with-age/updated_csv_with_age_category.xlsx"
FAULTS_SHP   = "/kaggle/input/datasets/atifbilal/active-faults-pk-globe/gem_active_faults.shp"
TECTONIC_SHP = "/kaggle/input/datasets/atifbilal/techtonicunits-pk/WEP_PRVG.SHP"

# Optional semantic text input for auxiliary normalization
SEMANTIC_TEXT_INLINE = ""
SEMANTIC_TEXT_PATH = os.getenv("LEGEND_OCR_TXT_PATH", "").strip()

# ============================================================
# OUTPUT PATHS
# ============================================================
MASTER_OUT_DIR = "/kaggle/working/LLMAG_master_clean"
TABLES_OUT_DIR = "/kaggle/working/LLMAG_tables_clean"

for _d in [MASTER_OUT_DIR, TABLES_OUT_DIR]:
    os.makedirs(_d, exist_ok=True)

# ============================================================
# AGENT / MODEL SETTINGS
# ============================================================
AGENT_STEPS = [
    "1. Read structured geological inputs",
    "2. Build control mapping table",
    "3. Optionally normalize auxiliary semantic text with DeepSeek",
    "4. Merge semantic support with control mapping",
    "5. Run tectonic-constrained GIS generalization",
    "6. Cluster copper occurrences and export outputs",
]

USE_DEEPSEEK = True
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "").strip()
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-chat").strip()
DEEPSEEK_TIMEOUT = 120

ROI_UTM_METRIC = "EPSG:32642"
CLUSTER_LABELS = ["A", "B", "C", "D", "E"]
K = 5

# ============================================================
# HELPERS
# ============================================================
def print_agent_trace():
    print("\n[LLMAG Agent Trace]")
    for s in AGENT_STEPS:
        print(" ", s)
    print("\nAgent role : task decomposition, step coordination, and workflow orchestration")
    print("LLM role   : optional semantic normalization")
    print("GIS role   : deterministic spatial generalization")

def ensure_epsg4326(gdf):
    if gdf is None or len(gdf) == 0:
        return gdf
    if gdf.crs is None:
        return gdf.set_crs("EPSG:4326", allow_override=True)
    if str(gdf.crs).upper() != "EPSG:4326":
        return gdf.to_crs("EPSG:4326")
    return gdf

def find_col_ci(df, candidates):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in lower_map:
            return lower_map[key]
    for cand in candidates:
        key = str(cand).strip().lower()
        for c in df.columns:
            cc = str(c).strip().lower()
            if key == cc or key in cc or cc in key:
                return c
    return None

def norm_abbr(x):
    return str(x).strip().lower()

def normalize_class_label(x):
    x = str(x).strip()
    return x if x else "Unknown"

def pick_tectonic_name_column(gdf):
    priority = [
        "TectonicUnit", "PRVG", "PRV_NAME", "PRVNAME", "PROV_NAME", "PROVNAME",
        "NAME", "Province", "PROVINCE", "UNIT", "Unit", "TECTONICUNIT",
        "prv_name", "prov_name", "provname", "name"
    ]
    cols = list(gdf.columns)
    for c in priority:
        if c in cols:
            return c
    obj_cols = [c for c in cols if c != "geometry" and gdf[c].dtype == "object"]
    if not obj_cols:
        raise KeyError(f"No suitable tectonic text column found. Columns: {cols[:40]}")
    return obj_cols[0]

def safe_json_loads(s):
    s = str(s).strip()
    try:
        return json.loads(s)
    except Exception:
        pass
    start = s.find("{")
    end = s.rfind("}")
    if start != -1 and end != -1 and end > start:
        return json.loads(s[start:end+1])
    raise ValueError("Could not parse JSON from model response.")

def read_optional_text(text_inline="", text_path=""):
    if str(text_inline).strip():
        return str(text_inline).strip()
    if str(text_path).strip() and os.path.exists(text_path):
        with open(text_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read().strip()
    return ""

def auto_utm_epsg_from_bounds(gdf4326):
    cen = gdf4326.geometry.unary_union.centroid
    lon = float(cen.x)
    lat = float(cen.y)
    zone = int(np.floor((lon + 180) / 6) + 1)
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return f"EPSG:{epsg}"

# ============================================================
# STEP 2: CONTROL TABLE
# ============================================================
def build_control_mapping(control_csv_path):
    df = pd.read_csv(control_csv_path)

    abbr_col = find_col_ci(df, ["Abbreviation", "abbr", "code", "unit", "name"])
    class_col = find_col_ci(df, [
        "GeneralizedClass", "FinalLabel", "Category", "Age_category",
        "Age Category", "class", "category"
    ])

    if abbr_col is None or class_col is None:
        raise ValueError(
            f"CONTROL_CSV must contain an abbreviation column and a class/category column. "
            f"Found: {list(df.columns)}"
        )

    out = df[[abbr_col, class_col]].copy()
    out.columns = ["Abbreviation", "ControlClass"]
    out["Abbreviation"] = out["Abbreviation"].astype(str).str.strip()
    out["ControlClass"] = out["ControlClass"].astype(str).map(normalize_class_label)
    out["_abbr_key"] = out["Abbreviation"].map(norm_abbr)
    out = out.drop_duplicates(subset=["_abbr_key"], keep="first").reset_index(drop=True)
    return out

# ============================================================
# STEP 3: OPTIONAL LLM NORMALIZATION
# ============================================================
def call_deepseek_for_mapping(semantic_text):
    if not DEEPSEEK_API_KEY:
        raise RuntimeError("DEEPSEEK_API_KEY is empty.")
    if requests is None:
        raise RuntimeError("requests is not available in this environment.")
    if not str(semantic_text).strip():
        raise RuntimeError("Semantic input text is empty.")

    system_prompt = """
You are a geological text normalization assistant.

Return exactly one valid JSON object and nothing else.

Task:
Read auxiliary geological text (legend text, abbreviated unit descriptions, or similar)
and convert it into a structured normalization table.

Do not perform GIS work.
Do not invent units not supported by the text.
Keep output conservative when text is noisy.

Output format:
{
  "records": [
    {
      "Abbreviation": "KJ",
      "NormalizedLabel": "Cretaceous",
      "Confidence": 0.92,
      "Notes": "short note"
    }
  ]
}

Rules:
- Return valid JSON only.
- Top-level key must be "records".
- "records" must be a list.
- If nothing useful is found, return {"records":[]}.
- Confidence must be between 0 and 1.
- Notes must be short plain text.
- Do not include markdown or explanations.
""".strip()

    user_prompt = f"""Normalize the following auxiliary geological text into JSON.

Text:
{semantic_text}
""".strip()

    payload = {
        "model": DEEPSEEK_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "response_format": {"type": "json_object"},
        "stream": False,
        "max_tokens": 2500,
    }

    url = f"{DEEPSEEK_BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }

    t0 = time()
    resp = requests.post(url, headers=headers, json=payload, timeout=DEEPSEEK_TIMEOUT)
    llm_seconds = time() - t0
    resp.raise_for_status()

    data = resp.json()
    content = data["choices"][0]["message"]["content"]
    obj = safe_json_loads(content)
    records = obj.get("records", [])

    if not isinstance(records, list):
        raise ValueError("DeepSeek JSON did not contain a list under 'records'.")

    df = pd.DataFrame(records)
    if len(df) == 0:
        print(f"  done -> 3. Optional semantic normalization ({llm_seconds:.2f}s, 0 records)")
        return pd.DataFrame(columns=["Abbreviation", "LLM_Label", "Confidence", "Notes", "_abbr_key"])

    for col in ["Abbreviation", "NormalizedLabel", "Confidence", "Notes"]:
        if col not in df.columns:
            df[col] = np.nan

    df["Abbreviation"] = df["Abbreviation"].astype(str).str.strip()
    df["LLM_Label"] = df["NormalizedLabel"].astype(str).map(normalize_class_label)
    df["Confidence"] = pd.to_numeric(df["Confidence"], errors="coerce").fillna(0.0).clip(0, 1)
    df["Notes"] = df["Notes"].astype(str).str.strip()
    df["_abbr_key"] = df["Abbreviation"].map(norm_abbr)
    df = df.drop_duplicates(subset=["_abbr_key"], keep="first").reset_index(drop=True)

    print(f"  done -> 3. Optional semantic normalization ({llm_seconds:.2f}s, {len(df)} records)")
    return df[["Abbreviation", "LLM_Label", "Confidence", "Notes", "_abbr_key"]]

# ============================================================
# STEP 4: MERGED FINAL MAPPING
# ============================================================
def build_final_mapping():
    print("  done -> 1. Read structured geological inputs")
    control_map = build_control_mapping(CONTROL_CSV)
    print("  done -> 2. Build control mapping table")

    semantic_text = read_optional_text(
        text_inline=SEMANTIC_TEXT_INLINE,
        text_path=SEMANTIC_TEXT_PATH
    )

    llm_used = False
    llm_error = ""

    if USE_DEEPSEEK and DEEPSEEK_API_KEY and str(semantic_text).strip():
        try:
            llm_map = call_deepseek_for_mapping(semantic_text)
            llm_used = True
        except Exception as e:
            llm_map = pd.DataFrame(columns=["Abbreviation", "LLM_Label", "Confidence", "Notes", "_abbr_key"])
            llm_error = str(e)
            print("  warning -> DeepSeek skipped/fallback:", llm_error)
    else:
        llm_map = pd.DataFrame(columns=["Abbreviation", "LLM_Label", "Confidence", "Notes", "_abbr_key"])
        print("  done -> 3. Optional semantic normalization (skipped)")

    final = control_map.merge(
        llm_map[["Abbreviation", "LLM_Label", "Confidence", "Notes", "_abbr_key"]],
        on="_abbr_key", how="left", suffixes=("", "_llm")
    )

    final["GeneralizedClass"] = np.where(
        final["LLM_Label"].notna() & (final["LLM_Label"].astype(str).str.strip() != ""),
        final["LLM_Label"],
        final["ControlClass"]
    )
    final["GeneralizedClass"] = final["GeneralizedClass"].astype(str).map(normalize_class_label)
    final["MappingSource"] = np.where(
        final["LLM_Label"].notna() & (final["LLM_Label"].astype(str).str.strip() != ""),
        "deepseek+control",
        "control"
    )

    keep_cols = [
        "Abbreviation", "ControlClass", "LLM_Label", "GeneralizedClass",
        "Confidence", "Notes", "MappingSource", "_abbr_key"
    ]
    final = final[keep_cols].drop_duplicates(subset=["_abbr_key"], keep="first").reset_index(drop=True)

    final_csv = os.path.join(MASTER_OUT_DIR, "unit_to_generalized_class_mapping.csv")
    final_xlsx = os.path.join(MASTER_OUT_DIR, "unit_to_generalized_class_mapping.xlsx")
    final.to_csv(final_csv, index=False)
    try:
        final.to_excel(final_xlsx, index=False)
    except Exception:
        pass

    meta = {
        "USE_DEEPSEEK": USE_DEEPSEEK,
        "DEEPSEEK_MODEL": DEEPSEEK_MODEL,
        "LLM_used": llm_used,
        "semantic_text_present": bool(str(semantic_text).strip()),
        "llm_error": llm_error,
    }
    with open(os.path.join(MASTER_OUT_DIR, "mapping_run_metadata.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    print("  done -> 4. Merge semantic support with control mapping")
    return final, meta

# ============================================================
# STEP 5 + 6: CORE GIS GENERALIZATION + CLUSTERING
# ============================================================
def build_core_objects():
    mapping_df, mapping_meta = build_final_mapping()
    abbr_to_class = dict(zip(mapping_df["_abbr_key"], mapping_df["GeneralizedClass"]))

    # --- geology ---
    gdf_geology = ensure_epsg4326(gpd.read_file(GEOLOGY_SHP))
    geo_unit_col = find_col_ci(
        gdf_geology,
        ["Name", "NAME", "Abbreviation", "abbr", "unit", "mapunit", "code"]
    )
    if geo_unit_col is None:
        raise ValueError(f"Could not find a geology unit column. Found: {list(gdf_geology.columns)}")

    gdf_geology["UnitCode"] = gdf_geology[geo_unit_col].astype(str).str.strip()
    gdf_geology["_abbr_key"] = gdf_geology["UnitCode"].map(norm_abbr)
    gdf_geology["GeneralizedClass"] = gdf_geology["_abbr_key"].map(abbr_to_class).fillna("Unknown")
    gdf_geology["GeneralizedClass"] = gdf_geology["GeneralizedClass"].astype(str).map(normalize_class_label)

    pak_gdf = gdf_geology.dissolve().reset_index(drop=True)[["geometry"]].copy()
    pak_gdf["geometry"] = pak_gdf["geometry"].buffer(0)

    # --- tectonics + faults ---
    gdf_tect = ensure_epsg4326(gpd.read_file(TECTONIC_SHP))
    gdf_faults = ensure_epsg4326(gpd.read_file(FAULTS_SHP))

    if "TectonicUnit" not in gdf_tect.columns:
        tect_name_col = pick_tectonic_name_column(gdf_tect)
        gdf_tect["TectonicUnit"] = gdf_tect[tect_name_col].astype(str).str.strip()
    else:
        gdf_tect["TectonicUnit"] = gdf_tect["TectonicUnit"].astype(str).str.strip()

    gdf_tectonic_pk = gpd.clip(gdf_tect[["TectonicUnit", "geometry"]], pak_gdf)
    gdf_faults_pk = gpd.clip(gdf_faults[["geometry"]], pak_gdf)

    # --- deterministic tectonic-constrained generalization ---
    geo_small = gdf_geology[["UnitCode", "GeneralizedClass", "geometry"]].copy()
    gdf_inter = gpd.overlay(
        geo_small,
        gdf_tectonic_pk[["TectonicUnit", "geometry"]],
        how="intersection",
        keep_geom_type=False
    )

    poly_types = {"Polygon", "MultiPolygon"}
    gdf_inter = gdf_inter[gdf_inter.geometry.type.isin(poly_types)].copy()
    gdf_inter = gdf_inter[gdf_inter.geometry.notnull()].copy()

    gdf_generalized = gdf_inter.dissolve(by=["TectonicUnit", "GeneralizedClass"]).reset_index()
    gdf_generalized = gdf_generalized[gdf_generalized.geometry.notnull()].copy()

    print("  done -> 5. Run tectonic-constrained GIS generalization")

    # --- copper points ---
    cu_df = pd.read_excel(COPPER_XLSX)

    major_col = find_col_ci(cu_df, ["Major Mineral", "major mineral", "Mineral", "commodity"])
    if major_col is not None:
        mask = cu_df[major_col].astype(str).str.strip().str.lower() == "copper"
        if mask.any():
            cu_df = cu_df[mask].copy()

    lon_col = find_col_ci(cu_df, ["Longitude", "Lon", "X"])
    lat_col = find_col_ci(cu_df, ["Latitude", "Lat", "Y"])
    if lon_col is None or lat_col is None:
        raise ValueError(f"Could not find longitude/latitude columns. Found: {list(cu_df.columns)}")

    cu_df["Longitude"] = pd.to_numeric(cu_df[lon_col], errors="coerce")
    cu_df["Latitude"] = pd.to_numeric(cu_df[lat_col], errors="coerce")
    cu_df = cu_df.dropna(subset=["Longitude", "Latitude"]).copy()

    copper_gdf = gpd.GeoDataFrame(
        cu_df,
        geometry=gpd.points_from_xy(cu_df["Longitude"], cu_df["Latitude"]),
        crs="EPSG:4326"
    )

    # --- GMM clustering ---
    if len(copper_gdf) >= K:
        coords = copper_gdf[["Longitude", "Latitude"]].values
        scaler = StandardScaler()
        X = scaler.fit_transform(coords)

        gmm = GaussianMixture(n_components=K, covariance_type="full", random_state=42)
        raw_lab = gmm.fit_predict(X)

        means_unscaled = scaler.inverse_transform(gmm.means_)
        order = np.argsort(means_unscaled[:, 0])  # west -> east
        remap = {old: new for new, old in enumerate(order)}

        copper_gdf["Cluster"] = pd.Series(raw_lab).map(remap).astype(int)
        copper_gdf["ClusterLabel"] = copper_gdf["Cluster"].map({i: CLUSTER_LABELS[i] for i in range(K)})
        copper_gdf["ClusterProb"] = gmm.predict_proba(X).max(axis=1)
    else:
        copper_gdf["Cluster"] = 0
        copper_gdf["ClusterLabel"] = "A"
        copper_gdf["ClusterProb"] = 1.0

    # --- exports ---
    generalized_gpkg = os.path.join(MASTER_OUT_DIR, "generalized_geology.gpkg")
    generalized_geojson = os.path.join(MASTER_OUT_DIR, "generalized_geology.geojson")
    copper_gpkg = os.path.join(MASTER_OUT_DIR, "copper_points_clustered.gpkg")

    gdf_generalized.to_file(generalized_gpkg, driver="GPKG")
    gdf_generalized.to_file(generalized_geojson, driver="GeoJSON")
    copper_gdf.to_file(copper_gpkg, driver="GPKG")

    # --- Summary  tables ---
    mapping_summary = (
    mapping_df[["GeneralizedClass", "MappingSource"]]
    .value_counts(dropna=False)
    .rename("n")
    .reset_index()
    )
    mapping_summary.to_csv(os.path.join(TABLES_OUT_DIR, "Mapping_summary.csv"), index=False)

    metric_crs = ROI_UTM_METRIC if ROI_UTM_METRIC else auto_utm_epsg_from_bounds(gdf_generalized)

    tectonic_area = gdf_generalized.to_crs(metric_crs).copy()
    tectonic_area["Area_km2"] = tectonic_area.geometry.area / 1e6

    table_tect = (
    tectonic_area.groupby("TectonicUnit", dropna=False)["Area_km2"]
    .sum()
    .reset_index()
    .sort_values("Area_km2", ascending=False)
    )
    table_tect.to_csv(os.path.join(TABLES_OUT_DIR, "Tectonic_area.csv"), index=False)

    print("  done -> 6. Cluster copper occurrences and export outputs")

    print("\n[Core outputs]")
    print("  Mapping rows        :", len(mapping_df))
    print("  Generalized polygons:", len(gdf_generalized))
    print("  Tectonic units      :", len(gdf_tectonic_pk))
    print("  Fault features      :", len(gdf_faults_pk))
    print("  Copper points       :", len(copper_gdf))
    print("  Semantic normalization source :", "DeepSeek + control table" if mapping_meta["LLM_used"] else "control table")
    if mapping_meta.get("llm_error"):
        print("  LLM fallback reason :", mapping_meta["llm_error"])

    print("\n[Saved files]")
    print(" ", os.path.join(MASTER_OUT_DIR, "unit_to_generalized_class_mapping.csv"))
    print(" ", generalized_gpkg)
    print(" ", generalized_geojson)
    print(" ", copper_gpkg)
    print(" ", os.path.join(TABLES_OUT_DIR, "Mapping summary.csv"))
    print(" ", os.path.join(TABLES_OUT_DIR, "Tectonic_area.csv"))
    print(" ", os.path.join(TABLES_OUT_DIR, "Copper_cluster_summary.csv"))

    return {
        "mapping_df": mapping_df,
        "mapping_meta": mapping_meta,
        "gdf_geology": gdf_geology,
        "pak_gdf": pak_gdf,
        "gdf_tectonic_pk": gdf_tectonic_pk,
        "gdf_faults_pk": gdf_faults_pk,
        "gdf_generalized": gdf_generalized,
        "copper_gdf": copper_gdf,
    }

# ============================================================
# CONTROLLER
# ============================================================
def run_llmag_master_pipeline():
    print_agent_trace()
    core = build_core_objects()
    globals().update(core)
    print("\nPipeline finished.")
    print("This notebook state now contains the prepared core objects for downstream figures and tables.")
    return core

# ============================================================
# ONE-CLICK RUN
# ============================================================
if __name__ == "__main__":
    core = run_llmag_master_pipeline()


[LLMAG Agent Trace]
  1. Read structured geological inputs
  2. Build control mapping table
  3. Optionally normalize auxiliary semantic text with DeepSeek
  4. Merge semantic support with control mapping
  5. Run tectonic-constrained GIS generalization
  6. Cluster copper occurrences and export outputs

Agent role : task decomposition, step coordination, and workflow orchestration
LLM role   : optional semantic normalization
GIS role   : deterministic spatial generalization
  done -> 1. Read structured geological inputs
  done -> 2. Build control mapping table
  done -> 3. Optional semantic normalization (skipped)
  done -> 4. Merge semantic support with control mapping
  done -> 5. Run tectonic-constrained GIS generalization
  done -> 6. Cluster copper occurrences and export outputs

[Core outputs]
  Mapping rows        : 163
  Generalized polygons: 88
  Tectonic units      : 11
  Fault features      : 199
  Copper points       : 90
  Semantic normalization source : control table

[